In [ ]:
import warnings
warnings.filterwarnings("ignore")
print("✅ Warnings suppressed")

✅ Warnings suppressed


In [ ]:
# Install required packages (run once per session)
!pip install -q --upgrade pip
!pip install -q "langchain<0.4" "langchain-community<0.4.2" "langchain-core<0.4"
!pip install -q langchain-groq langchain-huggingface ragas datasets python-dotenv sentence-transformers

print("✅ Packages installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 33.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
langgraph-prebuilt 1.1.0 requires langchain-core>=1.3.1, but you have langchain-core 0.3.86 which is incompatible.
langgraph 1.2.6 requires langchain-core<2,>=1.4.7, but you have langchain-core 0.3.86 which is incompatible.
langgraph-sdk 0.4.2 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.3.86 which is incompatible.
✅ Packages installed successfully with compatible versions


In [ ]:
import os
from dotenv import load_dotenv
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

print("✅ Imports successful")

In [ ]:
from google.colab import userdata

groq_api_key = userdata.get("GROQ_API_KEY")

if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found in Colab Secrets")

print("✅ Groq API key loaded from Colab Secrets")

✅ Environment variables loaded


In [ ]:
llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0,
    api_key=groq_api_key
)

embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

print("✅ LLM and Embeddings initialized")

✅ LLM initialized for RAGAS evaluation


In [ ]:
test_cases = [
    {
        "question": "楽天の2025年度の連結Non-GAAP営業利益はいくらでしたか？",
        "contexts": [
            "2025年度は、前年に引き続きすべてのセグメントで増収増益を達成しました。売上収益は前年比9.5%増の2兆4,966億円、Non-GAAP営業利益は前年比992億円の改善となる1,063億円を計上しました。",
            "連結売上収益 2.5 兆円 前年同期比 +9.5% （2025年通期）。Non-GAAP営業利益 1,063 億円 前年同期比 +992億円増益 （2025年通期）。"
        ],
        "answer": "1,063億円です。"
    },
    {
        "question": "楽天はAIをどのように活用していますか？",
        "contexts": [
            "楽天では現在、AIを使ってマーケティング効率、オペレーション効率、クライアント効率をそれぞれ20%改善する「トリプル20」という明確な目標があり、社内外でAIの導入に全面的に取り組んでいます。",
            "AI導入によるクリエイティブ作業時間の削減や、日常業務の支援、顧客体験の変革にも取り組んでいます。2025年12月には「楽天市場」においてエージェント型AIツール「Rakuten AI」を本格展開しました。"
        ],
        "answer": "楽天はAIをマーケティング効率、オペレーション効率、クライアント効率の向上や、顧客体験の変革、エージェント型AIの展開などに活用しています。"
    },
    {
        "question": "日本政府は2030年に対日直接投資残高をいくらに引き上げる目標を掲げていますか？",
        "contexts": [
            "日本政府は、2025年6月に公表した「対日直接投資促進プログラム2025」および「経済財政運営と改革の基本方針2025」の中で、2030年の対日直接投資残高の目標を100兆円から120兆円に引き上げ、さらに2030年代前半のできるだけ早期に150兆円を目指す方針を示した。"
        ],
        "answer": "120兆円"
    },
    {
        "question": "楽天モバイルと楽天エコシステムの関係について説明してください。",
        "contexts": [
            "「楽天モバイル」契約者は、非契約者と比較して、グループの各サービスにおいてより多くの金額をご利用いただいています。「楽天市場」では非契約者比で＋48.8％、「楽天トラベル」では＋19.6％、「楽天カード」では＋30.9％となっており、「楽天モバイル」と「楽天エコシステム」の結び付きの深さを示しています。",
            "「楽天モバイル」が、「楽天エコシステム」への利用拡張を促す強力な起点として機能していることが、この結果から見て取れます。"
        ],
        "answer": "楽天モバイルは楽天エコシステムの中核インフラであり、契約者は非契約者に比べて各サービスをより多く利用しています。"
    },
    {
        "question": "GX需要創出のために企業はどのような取り組みをすべきですか？",
        "contexts": [
            "特に初期需要創出にあたっては、GX製品を選択し、積極的に調達する企業の取組が不可欠であり、取組を通じた需要創出への貢献を見える化・評価することにより、企業による積極調達を後押しする枠組みが重要。"
        ],
        "answer": "GX製品を選択し、積極的に調達すること。"
    },
    {
        "question": "通商白書で述べられている日本の貿易上の主な課題は何ですか？",
        "contexts": [
            "本章では、①保護主義と貿易摩擦、②過剰生産能力と過剰依存のリスク、③地政学リスクと経済安全保障認識、④パワーバランスの変化とグローバルサウス、⑤デジタル化とグリーン移行への多様な対応に注目して、近年の動向と変化を概観する。"
        ],
        "answer": "・保護主義と貿易摩擦\n・過剰生産能力と過剰依存のリスク\n・地政学リスクと経済安全保障認識\n・パワーバランスの変化とグローバルサウス\n・デジタル化・グリーン移行への多様な対応"
    }
]

print(f"✅ Loaded {len(test_cases)} real test cases")

In [ ]:
eval_data = {
    "question": [case["question"] for case in test_cases],
    "contexts": [case["contexts"] for case in test_cases],
    "answer": [case["answer"] for case in test_cases]
}

dataset = Dataset.from_dict(eval_data)

print(f"✅ Dataset prepared with {len(test_cases)} test cases")

✅ Successfully prepared 6 real test cases for RAGAS
Sample question: 楽天の2025年度の連結Non-GAAP営業利益はいくらでしたか？ ...


In [ ]:
result = evaluate(
    dataset=dataset,
    metrics=[faithfulness, answer_relevancy],
    llm=llm,
    embeddings=embeddings
)

print("✅ RAGAS Evaluation Completed")
result

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

✅ Embeddings model loaded: BAAI/bge-m3


In [ ]:
result = evaluate(
    dataset=dataset,
    metrics=[faithfulness, answer_relevancy],
    llm=llm,
    embeddings=embeddings
)

print("✅ RAGAS Evaluation Completed using real test cases")
result

Evaluating:   0%|          | 0/12 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[4]: OutputParserException(Invalid json output: The output string did not satisfy the constraints given in the prompt. The correct output should be in a JSON format that complies with the specified schema. Given the input: 
    { 
        "question": "日本政府は2030年に対日直接投資残高をいくらに引き上げる目標を掲げていますか？", 
        "answer": "120 兆円" 
    } 
    The answer provided is a simple statement about a target value. To break it down into a fully understandable statement without pronouns, we can rephrase it to include the subject (日本政府, the Japanese government) for clarity. 
    Thus, adhering strictly to the format and ensuring clarity without adding unnecessary complexity, the output should be: 
    { 
        "statements": [ 
            "日本政府の2030年に対日直接投資残高の目標は120兆円です。" 
        ] 
    }
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE )
ERROR:ragas.executor:Exception raised in Job[1]: BadRequestError(Er

✅ RAGAS Evaluation Completed using real test cases


{'faithfulness': 0.9643, 'answer_relevancy': 0.7633}